In [1]:
import duckdb
import os
import unicodedata


In [2]:
RAW_DIR = "M26_DA_A2_Part1"
CLEAN_DIR = "cleaned"
SEED_DIR = "seed"  

In [4]:
os.makedirs(CLEAN_DIR, exist_ok=True)
os.makedirs(SEED_DIR, exist_ok=True)

files = os.listdir("M26_DA_A2_Part1")
print(files)

['categories.csv', 'products.csv', 'order_items.csv', 'closed_deals.csv', 'order_reviews.csv', 'geolocation.csv', 'sellers.csv', '.ipynb_checkpoints', 'leads.csv', 'order_payments.csv', 'orders.csv', 'customers.csv']


In [5]:
con = duckdb.connect()


In [6]:
def strip_accents_lower(s):
    if s is None:
        return None
    s = s.strip().lower()
    # Fix common mis-encoded characters seen in geolocation.csv (e.g. 'sa£o' -> 'sao')
    s = s.replace("£", "o").replace("§", "s")
    # Strip accents (São -> Sao)
    s = "".join(
        c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c)
    )
    return s
 
 
con.create_function("norm_city", strip_accents_lower, ["VARCHAR"], "VARCHAR")
 

In [7]:
print("Cleaning categories.csv ...")
con.execute(f"""
    COPY (
        SELECT
            trim(lower(product_category_name)) AS product_category_name,
            trim(lower(product_category_name_english)) AS product_category_name_english
        FROM read_csv_auto('{RAW_DIR}/categories.csv', encoding='utf-8')
    ) TO '{CLEAN_DIR}/categories_clean.csv' (HEADER, DELIMITER ',')
""")
 

Cleaning categories.csv ...


In [8]:
print("Cleaning geolocation.csv ...")
con.execute(f"""
    CREATE OR REPLACE TEMP TABLE geo_raw AS
    SELECT DISTINCT
        geolocation_zip_code_prefix AS zip_code_prefix,
        geolocation_lat AS lat,
        geolocation_lng AS lng,
        norm_city(geolocation_city) AS city,
        upper(trim(geolocation_state)) AS state
    FROM read_csv_auto('{RAW_DIR}/geolocation.csv')
    WHERE geolocation_lat BETWEEN -35 AND 6      -- drop obvious lat/lng outliers
      AND geolocation_lng BETWEEN -75 AND -30    -- outside Brazil's bounding box
""")
 
con.execute(f"""
    CREATE OR REPLACE TEMP TABLE geo_modal_city AS
    SELECT zip_code_prefix, city, state, cnt,
           ROW_NUMBER() OVER (PARTITION BY zip_code_prefix ORDER BY cnt DESC) AS rn
    FROM (
        SELECT zip_code_prefix, city, state, COUNT(*) AS cnt
        FROM geo_raw
        GROUP BY zip_code_prefix, city, state
    )
""")
 
con.execute(f"""
    COPY (
        SELECT
            g.zip_code_prefix,
            ROUND(AVG(r.lat), 6) AS lat,
            ROUND(AVG(r.lng), 6) AS lng,
            g.city,
            g.state
        FROM geo_modal_city g
        JOIN geo_raw r ON r.zip_code_prefix = g.zip_code_prefix
        WHERE g.rn = 1
        GROUP BY g.zip_code_prefix, g.city, g.state
    ) TO '{CLEAN_DIR}/geolocation_clean.csv' (HEADER, DELIMITER ',')
""")
 

Cleaning geolocation.csv ...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [9]:
print("Cleaning customers.csv ...")
con.execute(f"""
    COPY (
        SELECT
            customer_id,
            customer_unique_id,
            customer_zip_code_prefix AS zip_code_prefix,
            norm_city(customer_city) AS customer_city,
            upper(trim(customer_state)) AS customer_state
        FROM read_csv_auto('{RAW_DIR}/customers.csv')
    ) TO '{CLEAN_DIR}/customers_clean.csv' (HEADER, DELIMITER ',')
""")
 
print("Cleaning sellers.csv ...")
con.execute(f"""
    COPY (
        SELECT
            seller_id,
            seller_zip_code_prefix AS zip_code_prefix,
            norm_city(seller_city) AS seller_city,
            upper(trim(seller_state)) AS seller_state
        FROM read_csv_auto('{RAW_DIR}/sellers.csv')
    ) TO '{CLEAN_DIR}/sellers_clean.csv' (HEADER, DELIMITER ',')
""")
 
 

Cleaning customers.csv ...
Cleaning sellers.csv ...


In [10]:
print("Cleaning products.csv ...")
con.execute(f"""
    COPY (
        SELECT DISTINCT
            product_id,
            COALESCE(trim(lower(product_category_name)), 'unknown_category') AS product_category_name,
            product_name_lenght,
            product_description_lenght,
            product_photos_qty,
            product_weight_g,
            product_length_cm,
            product_height_cm,
            product_width_cm
        FROM read_csv_auto('{RAW_DIR}/products.csv')
    ) TO '{CLEAN_DIR}/products_clean.csv' (HEADER, DELIMITER ',')
""")


Cleaning products.csv ...


In [11]:
print("Cleaning orders.csv ...")
con.execute(f"""
    COPY (
        SELECT DISTINCT
            order_id,
            customer_id,
            trim(lower(order_status)) AS order_status,
            CAST(order_purchase_timestamp AS TIMESTAMP) AS order_purchase_timestamp,
            CAST(order_approved_at AS TIMESTAMP) AS order_approved_at,
            CAST(order_delivered_carrier_date AS TIMESTAMP) AS order_delivered_carrier_date,
            CAST(order_delivered_customer_date AS TIMESTAMP) AS order_delivered_customer_date,
            CAST(order_estimated_delivery_date AS TIMESTAMP) AS order_estimated_delivery_date
        FROM read_csv_auto('{RAW_DIR}/orders.csv')
    ) TO '{CLEAN_DIR}/orders_clean.csv' (HEADER, DELIMITER ',')
""")
 

Cleaning orders.csv ...


In [12]:
print("Cleaning order_items.csv ...")
con.execute(f"""
    COPY (
        SELECT DISTINCT
            order_id,
            order_item_id,
            product_id,
            seller_id,
            CAST(shipping_limit_date AS TIMESTAMP) AS shipping_limit_date,
            CAST(price AS DECIMAL(10,2)) AS price,
            CAST(freight_value AS DECIMAL(10,2)) AS freight_value
        FROM read_csv_auto('{RAW_DIR}/order_items.csv')
    ) TO '{CLEAN_DIR}/order_items_clean.csv' (HEADER, DELIMITER ',')
""")

Cleaning order_items.csv ...


In [13]:
print("Cleaning order_payments.csv ...")
con.execute(f"""
    COPY (
        SELECT
            order_id,
            SUM(CAST(payment_value AS DECIMAL(10,2))) AS total_payment_value,
            COUNT(*) AS payment_txn_count,
            MAX(payment_installments) AS max_installments,
            STRING_AGG(DISTINCT payment_type, '|') AS payment_types
        FROM read_csv_auto('{RAW_DIR}/order_payments.csv')
        GROUP BY order_id
    ) TO '{CLEAN_DIR}/order_payments_clean.csv' (HEADER, DELIMITER ',')
""")
 

Cleaning order_payments.csv ...


In [14]:
print("Cleaning order_reviews.csv ...")
con.execute(f"""
    COPY (
        SELECT review_id, order_id, review_score, review_creation_date, review_answer_timestamp
        FROM (
            SELECT
                review_id,
                order_id,
                review_score,
                CAST(review_creation_date AS TIMESTAMP) AS review_creation_date,
                CAST(review_answer_timestamp AS TIMESTAMP) AS review_answer_timestamp,
                ROW_NUMBER() OVER (
                    PARTITION BY order_id ORDER BY review_answer_timestamp DESC
                ) AS rn
            FROM read_csv_auto('{RAW_DIR}/order_reviews.csv')
        )
        WHERE rn = 1
    ) TO '{CLEAN_DIR}/order_reviews_clean.csv' (HEADER, DELIMITER ',')
""")
 

Cleaning order_reviews.csv ...


In [15]:
print("Cleaning leads.csv + closed_deals.csv -> seller_acquisition ...")
con.execute(f"""
    COPY (
        SELECT
            cd.seller_id,
            cd.mql_id,
            l.origin AS lead_origin,
            CAST(l.first_contact_date AS DATE) AS first_contact_date,
            CAST(cd.won_date AS TIMESTAMP) AS won_date,
            trim(lower(cd.business_segment)) AS business_segment,
            trim(lower(cd.lead_type)) AS lead_type,
            trim(lower(cd.business_type)) AS business_type,
            cd.declared_product_catalog_size,
            cd.declared_monthly_revenue
        FROM read_csv_auto('{RAW_DIR}/closed_deals.csv') cd
        LEFT JOIN read_csv_auto('{RAW_DIR}/leads.csv') l ON cd.mql_id = l.mql_id
    ) TO '{CLEAN_DIR}/seller_acquisition_clean.csv' (HEADER, DELIMITER ',')
""")
 

Cleaning leads.csv + closed_deals.csv -> seller_acquisition ...


In [16]:
mapping_csv = """state_code,state_name,region
AC,Acre,North
AP,Amapa,North
AM,Amazonas,North
PA,Para,North
RO,Rondonia,North
RR,Roraima,North
TO,Tocantins,North
AL,Alagoas,Northeast
BA,Bahia,Northeast
CE,Ceara,Northeast
MA,Maranhao,Northeast
PB,Paraiba,Northeast
PE,Pernambuco,Northeast
PI,Piaui,Northeast
RN,Rio Grande do Norte,Northeast
SE,Sergipe,Northeast
DF,Distrito Federal,Central-West
GO,Goias,Central-West
MT,Mato Grosso,Central-West
MS,Mato Grosso do Sul,Central-West
ES,Espirito Santo,Southeast
MG,Minas Gerais,Southeast
RJ,Rio de Janeiro,Southeast
SP,Sao Paulo,Southeast
PR,Parana,South
RS,Rio Grande do Sul,South
SC,Santa Catarina,South
"""

with open("seed/state_region_mapping.csv", "w") as f:
    f.write(mapping_csv)

print("Created seed/state_region_mapping.csv")

Created seed/state_region_mapping.csv


In [17]:
print("Copying state_region_mapping.csv ...")
con.execute(f"""
    COPY (
        SELECT * FROM read_csv_auto('{SEED_DIR}/state_region_mapping.csv')
    ) TO '{CLEAN_DIR}/state_region_mapping_clean.csv' (HEADER, DELIMITER ',')
""")
 

Copying state_region_mapping.csv ...


In [18]:
print("Patching geolocation_clean.csv for missing customer/seller zip prefixes ...")
con.execute(f"CREATE OR REPLACE TEMP TABLE geo_snapshot AS SELECT * FROM read_csv_auto('{CLEAN_DIR}/geolocation_clean.csv')")
con.execute(f"""
    COPY (
        SELECT zip_code_prefix, lat, lng, city, state
        FROM (
            SELECT *, ROW_NUMBER() OVER (PARTITION BY zip_code_prefix ORDER BY city) AS rn
            FROM (
                SELECT * FROM geo_snapshot
                UNION ALL
                SELECT DISTINCT
                    cu.zip_code_prefix, CAST(NULL AS DOUBLE), CAST(NULL AS DOUBLE),
                    cu.customer_city, cu.customer_state
                FROM read_csv_auto('{CLEAN_DIR}/customers_clean.csv') cu
                LEFT JOIN geo_snapshot g
                    ON g.zip_code_prefix = cu.zip_code_prefix
                WHERE g.zip_code_prefix IS NULL
                UNION ALL
                SELECT DISTINCT
                    se.zip_code_prefix, CAST(NULL AS DOUBLE), CAST(NULL AS DOUBLE),
                    se.seller_city, se.seller_state
                FROM read_csv_auto('{CLEAN_DIR}/sellers_clean.csv') se
                LEFT JOIN geo_snapshot g
                    ON g.zip_code_prefix = se.zip_code_prefix
                WHERE g.zip_code_prefix IS NULL
            )
        )
        WHERE rn = 1   -- collapse to one row per zip_code_prefix even if both a
                       -- customer and a seller independently patched the same gap
    ) TO '{CLEAN_DIR}/geolocation_clean.csv' (HEADER, DELIMITER ',')
""")
 

Patching geolocation_clean.csv for missing customer/seller zip prefixes ...


In [19]:
print("Patching categories_clean.csv for product categories missing a translation ...")
con.execute(f"CREATE OR REPLACE TEMP TABLE category_snapshot AS SELECT * FROM read_csv_auto('{CLEAN_DIR}/categories_clean.csv')")
con.execute(f"""
    COPY (
        SELECT product_category_name, product_category_name_english FROM (
            SELECT * FROM category_snapshot
            UNION ALL
            SELECT DISTINCT p.product_category_name, CAST(NULL AS VARCHAR)
            FROM read_csv_auto('{CLEAN_DIR}/products_clean.csv') p
            LEFT JOIN category_snapshot c
                ON c.product_category_name = p.product_category_name
            WHERE c.product_category_name IS NULL
        )
    ) TO '{CLEAN_DIR}/categories_clean.csv' (HEADER, DELIMITER ',')
""")
 

Patching categories_clean.csv for product categories missing a translation ...


In [20]:
print("\n--- Sanity checks ---")
checks = {
    "customers_clean": "customers_clean.csv",
    "sellers_clean": "sellers_clean.csv",
    "products_clean": "products_clean.csv",
    "orders_clean": "orders_clean.csv",
    "order_items_clean": "order_items_clean.csv",
    "order_payments_clean": "order_payments_clean.csv",
    "order_reviews_clean": "order_reviews_clean.csv",
    "geolocation_clean": "geolocation_clean.csv",
    "seller_acquisition_clean": "seller_acquisition_clean.csv",
}
for name, fname in checks.items():
    n = con.execute(f"SELECT COUNT(*) FROM read_csv_auto('{CLEAN_DIR}/{fname}')").fetchone()[0]
    print(f"{name}: {n} rows")
 
# Confirm no orders were dropped despite missing delivery date / review
orig_orders = con.execute(f"SELECT COUNT(*) FROM read_csv_auto('{RAW_DIR}/orders.csv')").fetchone()[0]
clean_orders = con.execute(f"SELECT COUNT(*) FROM read_csv_auto('{CLEAN_DIR}/orders_clean.csv')").fetchone()[0]
print(f"\norders.csv row count preserved: {orig_orders} raw -> {clean_orders} cleaned "
      f"({'OK' if orig_orders == clean_orders else 'MISMATCH - investigate'})")
 
print("\nDone. Cleaned files written to:", CLEAN_DIR)


--- Sanity checks ---
customers_clean: 99441 rows
sellers_clean: 3095 rows
products_clean: 32951 rows
orders_clean: 99441 rows
order_items_clean: 112650 rows
order_payments_clean: 99440 rows
order_reviews_clean: 98673 rows
geolocation_clean: 19174 rows
seller_acquisition_clean: 842 rows

orders.csv row count preserved: 99441 raw -> 99441 cleaned (OK)

Done. Cleaned files written to: cleaned
